# Phase 5 — Churn drivers

**Question:** *why* do members churn — which factors move the probability, and by how much?

**Setup**
* Population: KKBox label population (memberships expiring March 2017) with history before the cutoff.
* Features: only information dated **on or before 28 Feb 2017** (`mart_churn_features`, enforced by a dbt test).
* **Model A** — all members, transaction + member features (complete data).
* **Model B** — members in the **10% user log sample** only, adds listening features. ⚠️ SAMPLED.
* Split: stratified random 70/30, seed 42. Preprocessing (top payment methods, median age) is learned on the training set only.
* Gender and city are deliberately excluded: a retention offer shouldn't be targeted on them, and they would not be actionable.

Logistic regression is the main model because its coefficients translate into plain English. A depth-3 decision tree is a cross-check that shows interactions as readable rules.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from streampulse.db import query, t, save_result, banner, RESULTS_DIR, IS_FIXTURE

banner()
FIG = RESULTS_DIR / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.tree import DecisionTreeClassifier, export_text

SEED = 42
CUTOFF = pd.Timestamp("2017-02-28")

In [ ]:
df = query(f"select * from {t('mart_churn_features')} where has_history_before_cutoff")
df["y"] = df["is_churn"].astype(int)
assert pd.to_datetime(df.last_period_start_date).max() <= CUTOFF, "LEAKAGE: a feature uses data after the cutoff"
print(f"{len(df):,} members, churn rate {df.y.mean():.2%}, in log sample: {df.in_log_sample.mean():.1%}")

## 5.1 Features
Units are chosen so each coefficient reads naturally ("per 10 NTD", "per extra active week-day").

In [ ]:
LABELS = {  # feature -> (plain-English name, unit of one step)
    "auto_renew_on": ("Auto-renew switched on", "on vs off"),
    "discounted": ("Paid below list price", "yes vs no"),
    "cancelled_before_cutoff": ("Already cancelled before the window", "yes vs no"),
    "monthly_price_per_10": ("Monthly-equivalent price", "+10 NTD"),
    "tenure_years": ("Tenure", "+1 year"),
    "reactivations": ("Past comebacks after churning", "+1 comeback"),
    "age_filled_per_10": ("Age", "+10 years"),
    "age_missing": ("Age not given", "missing vs given"),
    "active_days_30d_per_7": ("Active listening days, last 30d", "+7 days"),
    "listen_hours_30d_log": ("Listening time, last 30d", "x2.7 hours (log)"),
    "completion_rate_30d": ("Share of songs played to 75%+", "0 -> 1"),
    "active_days_change_per_7": ("Change in active days vs prior 30d", "+7 days"),
    "days_since_last_listen_per_7": ("Days since last listen", "+7 days"),
}


MIN_CELL = 5 if IS_FIXTURE else 20   # min churners AND non-churners (train) for a category to get its own coefficient


def fit_prep(train):
    """Learn preprocessing on TRAIN only (no information from the test set).

    Sparse categories are folded into 'other': with too few churners (or none) a
    logistic regression cannot estimate a coefficient (quasi-separation -> huge errors).
    """
    def keep_levels(col, candidates):
        ct = pd.crosstab(train[col], train.y)
        return [c for c in candidates if c in ct.index and ct.loc[c].min() >= MIN_CELL]
    methods = train.last_payment_method_id.value_counts().head(5).index.tolist()
    plans = train.last_plan_length_bucket.fillna("unknown").value_counts().index.tolist()
    return {"ref_method": methods[0], "top_methods": keep_levels("last_payment_method_id", methods[1:]),
            "plans": keep_levels("last_plan_length_bucket", [p for p in plans if p != "02. ~1 month"]),
            "age_median": train.age.median()}


def build_X(d, prep, with_logs):
    X = pd.DataFrame(index=d.index)
    X["auto_renew_on"] = d.last_is_auto_renew.astype(float)
    X["discounted"] = d.last_is_discounted.astype(float)
    X["cancelled_before_cutoff"] = d.last_period_cancelled_before_cutoff.fillna(False).astype(float)
    X["monthly_price_per_10"] = d.last_monthly_price_ntd.fillna(0) / 10
    X["tenure_years"] = d.tenure_days.clip(lower=0) / 365
    X["reactivations"] = d.n_reactivations.clip(upper=3)
    X["age_missing"] = d.age.isna().astype(float)
    X["age_filled_per_10"] = d.age.fillna(prep["age_median"]) / 10
    plan = d.last_plan_length_bucket.fillna("unknown")
    for b in prep["plans"]:                          # reference category = monthly plan
        X[f"plan_{b}"] = (plan == b).astype(float)
    X["plan_other"] = (~plan.isin(prep["plans"] + ["02. ~1 month"])).astype(float)
    pm = d.last_payment_method_id
    for m in prep["top_methods"]:                    # reference = most common method
        X[f"payment_method_{int(m)}"] = (pm == m).astype(float)
    X["payment_method_other"] = (~pm.isin(prep["top_methods"] + [prep["ref_method"]])).astype(float)
    if with_logs:
        X["active_days_30d_per_7"] = d.active_days_30d / 7
        X["listen_hours_30d_log"] = np.log1p(d.listen_minutes_30d / 60)
        X["completion_rate_30d"] = d.completion_rate_30d.fillna(0)
        X["active_days_change_per_7"] = d.active_days_change / 7
        X["days_since_last_listen_per_7"] = d.days_since_last_listen / 7
    return X.astype(float)


def drop_unestimable(X, y):
    """Drop constant columns and binary columns with < MIN_CELL cases in any class (train only)."""
    keep = []
    for c in X.columns:
        if X[c].std() == 0:
            print(f"  dropped {c}: constant"); continue
        if set(X[c].unique()) <= {0.0, 1.0}:
            ct = pd.crosstab(X[c], y)
            if ct.shape != (2, 2) or ct.values.min() < MIN_CELL:
                print(f"  dropped {c}: fewer than {MIN_CELL} cases in a cell"); continue
        keep.append(c)
    return X[keep]

In [ ]:
def fit_logit(X, y):
    Xc = sm.add_constant(X, has_constant="add")
    try:
        return sm.Logit(y.astype(int), Xc).fit(disp=0, maxiter=300)
    except Exception as e:                       # e.g. perfect separation / singular matrix
        print("Newton failed (", e, ") -> using BFGS")
        return sm.Logit(y.astype(int), Xc).fit(disp=0, maxiter=2000, method="bfgs")


def evaluate(model, X, y, name):
    p = model.predict(sm.add_constant(X, has_constant="add"))
    order = np.argsort(-p.values)
    top = order[: max(1, len(order) // 10)]
    out = {"model": name, "n_test": int(len(y)), "test_churn_rate": float(y.mean()),
           "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p)),
           "top_decile_churn_rate": float(y.values[top].mean()),
           "top_decile_share_of_churners": float(y.values[top].sum() / y.sum())}
    out["top_decile_lift"] = out["top_decile_churn_rate"] / out["test_churn_rate"]
    return out


def run(d, with_logs, name):
    tr, te = train_test_split(d, test_size=0.3, stratify=d.y, random_state=SEED)
    prep = fit_prep(tr)
    Xtr = drop_unestimable(build_X(tr, prep, with_logs), tr.y)
    Xte = build_X(te, prep, with_logs).reindex(columns=Xtr.columns, fill_value=0)
    m = fit_logit(Xtr, tr.y)
    return m, evaluate(m, Xte, te.y, name), (Xtr, tr.y, Xte, te.y, prep)

## 5.2 Model A — all members, no listening data

In [ ]:
mA, evA, dataA = run(df, with_logs=False, name="A: all members, no logs")
print(mA.summary2().tables[1].round(4).to_string())
evA

## 5.3 Model B — log-sample members, with listening behaviour ⚠️ 10% user sample

In [ ]:
dfB = df[df.in_log_sample].copy()
mB, evB, dataB = run(dfB, with_logs=True, name="B: log sample, with logs")
# Fair comparison: Model A's features on exactly the same sampled members
_, evA_same, _ = run(dfB, with_logs=False, name="A-features on log sample")
print(mB.summary2().tables[1].round(4).to_string())
pd.DataFrame([evA, evA_same, evB])

## 5.4 From coefficients to business statements
*Odds ratio* = how the odds of churning multiply per unit step. *Average marginal effect (AME)* = change in churn **probability**, in percentage points, averaged over members — easier for a business audience.

In [ ]:
def translate(model, X_train):
    ame = model.get_margeff(at="overall", dummy=True).summary_frame()
    coef = model.params.drop("const")
    ci = model.conf_int().drop("const")
    t_ = pd.DataFrame({"odds_ratio": np.exp(coef), "or_ci_low": np.exp(ci[0]), "or_ci_high": np.exp(ci[1]),
                       "p_value": model.pvalues.drop("const"), "ame_pp": ame["dy/dx"] * 100})
    t_["significant_5pct"] = t_.p_value < 0.05
    return t_.sort_values("ame_pp", key=abs, ascending=False)


def label(feat):
    if feat in LABELS:
        return LABELS[feat]
    if feat.startswith("plan_"):
        return (f"Plan '{feat[5:]}'", "vs a ~1 month plan")
    if feat.startswith("payment_method_"):
        return (f"Payment method {feat[15:]}", f"vs the most common method ({int(dataA[4]['ref_method'])})")
    return (feat.replace("_", " "), "yes vs no")


def sentence(feat, r):
    name, unit = label(feat)
    verb = "raises" if r.ame_pp > 0 else "lowers"
    return (f"{name} ({unit}) {verb} churn probability by about {abs(r.ame_pp):.1f} percentage points "
            f"(odds x{r.odds_ratio:.2f}, 95% CI {r.or_ci_low:.2f}-{r.or_ci_high:.2f}), holding the other factors fixed.")


drivers_A, drivers_B = translate(mA, dataA[0]), translate(mB, dataB[0])
display(drivers_A.round(4))
display(drivers_B.round(4))
LISTENING = ["active_days_30d_per_7", "listen_hours_30d_log", "completion_rate_30d",
             "active_days_change_per_7", "days_since_last_listen_per_7"]
for title, rows in [("Plain-English drivers (Model A, significant at 5%)", drivers_A[drivers_A.significant_5pct].head(6)),
                    ("Listening drivers (Model B, significant at 5%)",
                     drivers_B[drivers_B.significant_5pct & drivers_B.index.isin(LISTENING)])]:
    print("\n" + title + ":")
    if rows.empty:
        print(" - none significant at the 5% level")
    for f, r in rows.iterrows():
        print(" -", sentence(f, r))

## 5.5 Cross-check: depth-3 decision tree
The tree finds interactions the linear model can't, and its rules are readable. `class_weight='balanced'` stops it predicting "no churn" for everyone.

In [ ]:
Xtr, ytr, Xte, yte, _ = dataA
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=max(20, int(0.02 * len(Xtr))),
                              class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
tree_auc = roc_auc_score(yte, tree.predict_proba(Xte)[:, 1])
print(f"Tree test ROC AUC: {tree_auc:.3f}  (logistic A: {evA['roc_auc']:.3f})")
print(export_text(tree, feature_names=list(Xtr.columns), decimals=2))

## 5.6 Leakage & validity checklist
* ✅ Features ≤ 28 Feb 2017; labels are about March expiries → April behaviour.
* ✅ Preprocessing learned on train only.
* ✅ Test set never used for fitting or choosing features.
* ⚠️ Single label month → a random split, not a time-based one. A second label month (KKBox's older `train.csv`, Feb 2017) would allow true out-of-time validation — listed as a limitation.
* ⚠️ Coefficients are associations. *Auto-renew lowers churn* does not prove that switching someone's auto-renew on keeps them — that needs an experiment (Phase 6).

In [ ]:
def top_drivers(d, n=6):
    rows = []
    for f, r in d[d.significant_5pct].head(n).iterrows():
        rows.append({"feature": f, "plain_english": sentence(f, r), "ame_pp": float(r.ame_pp),
                     "odds_ratio": float(r.odds_ratio), "p_value": float(r.p_value)})
    return rows


res = {
    "population": {"members": int(len(df)), "churn_rate": float(df.y.mean()),
                   "log_sample_members": int(len(dfB)), "log_sample_churn_rate": float(dfB.y.mean())},
    "model_A_all_members": evA,
    "model_A_features_on_log_sample": evA_same,
    "model_B_log_sample_with_listening": evB,
    "auc_gain_from_listening_features": evB["roc_auc"] - evA_same["roc_auc"],
    "decision_tree_auc": float(tree_auc),
    "top_drivers_model_A": top_drivers(drivers_A),
    "top_drivers_model_B": top_drivers(drivers_B),
}
pd.concat({"A": drivers_A, "B": drivers_B}).to_csv(RESULTS_DIR / "phase05_driver_table.csv")
save_result("phase05_churn_drivers", res)
res